# ML Zoomcamp 2026 — Homework 8

Official: [08-deep-learning homework](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/08-deep-learning/homework.md) (do not edit).
Data: `data/` from `straight-curly-data/data.zip` (800 train, 201 held-out). `curly=0`, `straight=1`.
Write-up: `ML_08_HW.md`.

Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.
Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw08


## 0) Imports


In [1]:
import os
# os: the process environment. purpose: keep caches on E: before libraries start. at this point: first lines of the notebook.
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
# setdefault: write the variable only when it is missing. purpose: Matplotlib config stays off C:. at this point: before any import that reads it.
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")

import json
import random
import numpy as np
import torch
# torch: tensor and autograd library. purpose: define and train the hair model. at this point: cache paths are already set.
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
# random.seed: fix Python's RNG. purpose: later flips and crops stay repeatable. at this point: before datasets, loaders, and the model.
np.random.seed(SEED)
torch.manual_seed(SEED)
# manual_seed: fix PyTorch's RNG. purpose: same initial weights on every run. at this point: before HairModel() is built.
torch.use_deterministic_algorithms(True)
# use_deterministic_algorithms: pick stable math kernels. purpose: CPU runs do not drift. at this point: before any convolution.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(2)
# set_num_threads: use 2 CPU threads. purpose: match the course reference run. at this point: before data loading.
device = torch.device("cpu")
# device: where tensors live. purpose: the graded run is CPU. at this point: used by every .to(device) below.


## 1) Load data and preprocessing

Baseline training and the held-out set share one transform. Do not augment the held-out images.


In [2]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    # Resize: force every photo to 200x200. purpose: the linear layer expects 99x99 maps. at this point: before pixels become a tensor.
    transforms.ToTensor(),
    # ToTensor: image to a float tensor in [0, 1], channel first. purpose: the model cannot read a PIL image. at this point: after the resize.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # Normalize: ImageNet mean and std. purpose: put pixels on the scale this CNN expects. at this point: last step of the shared transform.
])
# Compose: run those steps in order. purpose: one transform object for ImageFolder. at this point: baseline train and held-out both use it.

train_dataset = datasets.ImageFolder("data/train", transform=evaluation_transform)
# ImageFolder: one subfolder per class. purpose: load the fixed 800 training images. at this point: curly sorts before straight, so curly=0.
evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transform)
# ImageFolder on data/test: the fixed 201-image held-out split. purpose: score every epoch without a new random split. at this point: same transform, no augmentation.

train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
# DataLoader: batches of 20. purpose: shuffle training with a fixed generator. at this point: seed 42, workers 0, before the model exists.
# Generator.manual_seed: private shuffle RNG. purpose: batch order does not follow other torch calls. at this point: baseline loader only.
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)
# DataLoader: held-out batches, no shuffle. purpose: evaluation order stays fixed. at this point: reused for both training phases.

print(len(train_dataset), len(evaluation_dataset), train_dataset.class_to_idx)


800 201 {'curly': 0, 'straight': 1}


## 2) Model architecture, loss, and optimizer


In [3]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        # Conv2d: 3x3 convolution, 3 channels to 32 filters. purpose: detect local hair edges. at this point: 200x200 becomes 198x198.
        self.pool = nn.MaxPool2d(kernel_size=2)
        # MaxPool2d: keep the strongest value in each 2x2 block. purpose: 198x198 becomes 99x99. at this point: right after the convolution.
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        # Linear: dense layer into 64 units. purpose: mix the flattened feature map. at this point: input size is fixed by pool.
        self.output = nn.Linear(64, 1)
        # Linear: one unit. purpose: emit a single logit. at this point: no sigmoid in the module.

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        # relu: set negative values to 0. purpose: nonlinearity after the convolution. at this point: before pooling.
        x = x.flatten(start_dim=1)
        # flatten: one vector per image. purpose: the dense layer needs a vector. at this point: the batch axis stays.
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)
        # output: raw logit, shape [batch, 1]. purpose: BCEWithLogitsLoss applies the sigmoid. at this point: end of forward.

model = HairModel().to(device)
# to: move weights onto CPU. purpose: match the graded device. at this point: model has just been created.
criterion = nn.BCEWithLogitsLoss()
# BCEWithLogitsLoss: sigmoid plus binary cross-entropy, in one stable step. purpose: train a single logit. at this point: before any epoch.
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
# SGD: stochastic gradient descent with momentum 0.8. purpose: the homework optimizer. at this point: this same object continues into the augmented phase.


## Q1 — Loss and output contract

Which loss matches the model above?

- `nn.MSELoss()`
- `nn.BCEWithLogitsLoss()`
- `nn.CrossEntropyLoss()`
- `nn.CosineEmbeddingLoss()`

Answer: **`nn.BCEWithLogitsLoss()`**


In [4]:
dummy_x = torch.zeros((2, 3, 200, 200), device=device)
dummy_logits = model(dummy_x)
# Calling the model runs forward. purpose: check the output shape. at this point: untrained weights, two fake images.
print("Output shape:", tuple(dummy_logits.shape))
print("Loss function:", type(criterion).__name__)


Output shape: (2, 1)
Loss function: BCEWithLogitsLoss


## Q2 — Parameter count

What is the total number of trainable parameters? Count biases as well and use `sum(p.numel() for p in model.parameters())`.

- `896`
- `11214912`
- `15896912`
- `20073473`

Answer: **`20073473`**


In [5]:
total_params = sum(p.numel() for p in model.parameters())
# numel: count of numbers in one parameter tensor. purpose: add weights and biases. at this point: before training; the count does not change later.
print("Total trainable parameters:", total_params)


Total trainable parameters: 20073473


## Q3 — Baseline accuracy

Train for 10 epochs with the baseline transform. Each epoch also scores the held-out split. Save `history_baseline.json` as four lists. Median of `train_accuracy`, rounded to two decimal places.

Answer: **`0.81`**


In [6]:
def run_phase(model, train_loader, evaluation_loader, optimizer, epochs=10):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": [],
    }
    for epoch in range(epochs):
        model.train()
        # train: turn on training behavior. purpose: this pass may update weights. at this point: start of the epoch.
        loss_sum = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            # unsqueeze: make labels [batch, 1]. purpose: match the logit shape inside the loss. at this point: each training batch.
            optimizer.zero_grad()
            # zero_grad: clear old gradients. purpose: this batch must not add to the previous one. at this point: before forward.
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            # backward: fill each parameter with a gradient. purpose: know which way to step. at this point: after the batch loss.
            optimizer.step()
            # step: apply SGD with momentum. purpose: update weights. at this point: immediately after backward.
            loss_sum += loss.item() * images.size(0)
            preds = torch.sigmoid(logits) >= 0.5
            # sigmoid: turn the logit into a probability. purpose: accuracy uses a 0.5 cutoff, not the loss. at this point: after the weight update.
            correct += (preds == labels.bool()).sum().item()
            total += images.size(0)

        model.eval()
        # eval: turn off training-only behavior. purpose: score the held-out set as inference. at this point: training batches for this epoch are done.
        eval_loss_sum = 0.0
        eval_correct = 0
        eval_total = 0
        with torch.no_grad():
            # no_grad: do not store gradients. purpose: evaluation must not change weights. at this point: held-out pass only.
            for images, labels in evaluation_loader:
                images = images.to(device)
                labels = labels.to(device).float().unsqueeze(1)
                logits = model(images)
                loss = criterion(logits, labels)
                eval_loss_sum += loss.item() * images.size(0)
                preds = torch.sigmoid(logits) >= 0.5
                eval_correct += (preds == labels.bool()).sum().item()
                eval_total += images.size(0)

        history["train_loss"].append(loss_sum / total)
        history["train_accuracy"].append(correct / total)
        history["evaluation_loss"].append(eval_loss_sum / eval_total)
        history["evaluation_accuracy"].append(eval_correct / eval_total)
        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Train Loss: {history['train_loss'][-1]:.4f} | "
            f"Train Acc: {history['train_accuracy'][-1]:.4f} | "
            f"Eval Loss: {history['evaluation_loss'][-1]:.4f} | "
            f"Eval Acc: {history['evaluation_accuracy'][-1]:.4f}"
        )
    return history

history_baseline = run_phase(model, train_loader, evaluation_loader, optimizer)
# run_phase: 10 epochs plus a held-out score each epoch. purpose: build the baseline history. at this point: first time the weights change.

with open("history_baseline.json", "w", encoding="utf-8") as f:
    json.dump(history_baseline, f, indent=2)
    # dump: write the four lists. purpose: Q3 and Q4 read this file. at this point: baseline phase is finished.
    f.write("\n")
print("Saved history_baseline.json")


Epoch 1/10 - Train Loss: 0.6385 | Train Acc: 0.6450 | Eval Loss: 0.6010 | Eval Acc: 0.6567
Epoch 2/10 - Train Loss: 0.5441 | Train Acc: 0.7175 | Eval Loss: 0.6003 | Eval Acc: 0.6517
Epoch 3/10 - Train Loss: 0.4955 | Train Acc: 0.7550 | Eval Loss: 0.6405 | Eval Acc: 0.6219
Epoch 4/10 - Train Loss: 0.4330 | Train Acc: 0.7913 | Eval Loss: 0.6274 | Eval Acc: 0.6915
Epoch 5/10 - Train Loss: 0.4048 | Train Acc: 0.8087 | Eval Loss: 0.6293 | Eval Acc: 0.6617
Epoch 6/10 - Train Loss: 0.3953 | Train Acc: 0.8075 | Eval Loss: 0.6332 | Eval Acc: 0.7114
Epoch 7/10 - Train Loss: 0.3437 | Train Acc: 0.8363 | Eval Loss: 0.6895 | Eval Acc: 0.6716
Epoch 8/10 - Train Loss: 0.2806 | Train Acc: 0.8850 | Eval Loss: 0.6497 | Eval Acc: 0.7214
Epoch 9/10 - Train Loss: 0.2236 | Train Acc: 0.9100 | Eval Loss: 0.7114 | Eval Acc: 0.7164
Epoch 10/10 - Train Loss: 0.1584 | Train Acc: 0.9387 | Eval Loss: 1.0212 | Eval Acc: 0.6368
Saved history_baseline.json


In [7]:
with open("history_baseline.json", encoding="utf-8") as f:
    history_baseline = json.load(f)
    # load: read the JSON back. purpose: the answer comes from the file, not a leftover list. at this point: after the baseline save.
median_acc = round(float(np.median(history_baseline["train_accuracy"])), 2)
# median: middle value of the 10 train accuracies. purpose: Q3. at this point: ten baseline epochs are in the file.
print("Median of train accuracy:", median_acc)


Median of train accuracy: 0.81


## Q4 — Baseline loss

Population standard deviation of `train_loss` in `history_baseline.json` (`np.std(..., ddof=0)`), rounded to three decimal places.

Answer: **`0.139`**


In [8]:
std_loss = round(float(np.std(history_baseline["train_loss"], ddof=0)), 3)
# std: population standard deviation (ddof=0). purpose: Q4. at this point: the ten baseline train losses are loaded.
print("Population std of train loss:", std_loss)


Population std of train loss: 0.139


## Q5 — Augmented evaluation loss

Keep the same model and optimizer. Replace only the training transform, train 10 more epochs, and save those epochs to `history_augmented.json`. Mean of `evaluation_loss`, rounded to three decimal places.

Answer: **`0.575`**


In [9]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    # RandomRotation: rotate up to 50 degrees with nearest-neighbor. purpose: vary hair angle on train images only. at this point: augmented phase.
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    # RandomResizedCrop: crop and resize back to 200. purpose: small shifts of the frame. at this point: after rotation, still training only.
    transforms.RandomHorizontalFlip(),
    # RandomHorizontalFlip: mirror left-right half the time. purpose: more pose variety. at this point: before ToTensor.
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset_aug = datasets.ImageFolder("data/train", transform=train_transform_augmented)
# ImageFolder again: same 800 files, new transform. purpose: augment training without touching the held-out set. at this point: model and optimizer are the ones from the baseline.
train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED + 1),
)
# manual_seed(SEED + 1): shuffle seed 43. purpose: a second fixed order, not a fresh model. at this point: just before the extra 10 epochs.

history_augmented = run_phase(model, train_loader_aug, evaluation_loader, optimizer)
# run_phase again: 10 more epochs on the same weights. purpose: the augmented history is only this second phase. at this point: baseline training is already finished.

with open("history_augmented.json", "w", encoding="utf-8") as f:
    json.dump(history_augmented, f, indent=2)
    f.write("\n")
print("Saved history_augmented.json")


Epoch 1/10 - Train Loss: 0.6758 | Train Acc: 0.6050 | Eval Loss: 0.6552 | Eval Acc: 0.6716
Epoch 2/10 - Train Loss: 0.6129 | Train Acc: 0.6475 | Eval Loss: 0.6608 | Eval Acc: 0.6269
Epoch 3/10 - Train Loss: 0.5899 | Train Acc: 0.6800 | Eval Loss: 0.5787 | Eval Acc: 0.7114
Epoch 4/10 - Train Loss: 0.5923 | Train Acc: 0.6750 | Eval Loss: 0.5679 | Eval Acc: 0.7114
Epoch 5/10 - Train Loss: 0.5604 | Train Acc: 0.7137 | Eval Loss: 0.5680 | Eval Acc: 0.7214
Epoch 6/10 - Train Loss: 0.5567 | Train Acc: 0.7087 | Eval Loss: 0.5510 | Eval Acc: 0.7015
Epoch 7/10 - Train Loss: 0.5441 | Train Acc: 0.7025 | Eval Loss: 0.5288 | Eval Acc: 0.7413
Epoch 8/10 - Train Loss: 0.5329 | Train Acc: 0.7212 | Eval Loss: 0.5401 | Eval Acc: 0.7114
Epoch 9/10 - Train Loss: 0.5019 | Train Acc: 0.7612 | Eval Loss: 0.5631 | Eval Acc: 0.7363
Epoch 10/10 - Train Loss: 0.5312 | Train Acc: 0.7262 | Eval Loss: 0.5404 | Eval Acc: 0.7313
Saved history_augmented.json


In [10]:
with open("history_augmented.json", encoding="utf-8") as f:
    history_augmented = json.load(f)
mean_eval_loss = round(float(np.mean(history_augmented["evaluation_loss"])), 3)
# mean: average of the 10 augmented evaluation losses. purpose: Q5. at this point: the second phase file is loaded.
print("Mean of evaluation loss:", mean_eval_loss)


Mean of evaluation loss: 0.575


## Q6 — Augmented evaluation accuracy

Mean of the last five `evaluation_accuracy` values (epochs 6 through 10 of the augmented phase), rounded to two decimal places.

Answer: **`0.72`**


In [11]:
last_five_mean = round(float(np.mean(history_augmented["evaluation_accuracy"][-5:])), 2)
# [-5:]: epochs 6-10 of the augmented phase. purpose: Q6. at this point: evaluation_accuracy already has 10 numbers.
print("Mean of last five evaluation accuracies:", last_five_mean)


Mean of last five evaluation accuracies: 0.72
